# Gemma 4 — Basic Usage (MLX)

## Imports

In [1]:
from pprint import pprint

import mlx_lm
import mlx_vlm

print("mlx-vlm:", mlx_vlm.__version__)

mlx-vlm: 0.6.17


## Load Model and Processor

In [2]:
MODEL_ID = "mlx-community/gemma-4-e2b-it-mxfp8"

model, processor = mlx_vlm.load(MODEL_ID)

print(f"Architecture: {model.config.model_type}")
print(f"Parameters: {mlx_lm.utils.get_total_parameters(model):,}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Architecture: gemma4
Parameters: 5,101,150,240


## Single Turn Generation

In [3]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<bos><|turn>user
What is the capital of France?<turn|>
<|turn>model



In [4]:
result = mlx_vlm.generate(model, processor, chat, max_tokens=128)
print(result.text)

The capital of France is **Paris**.


## System Prompt

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<bos><|turn>system
You are a helpful assistant who responds in all capitals. <turn|>
<|turn>user
What is the capital of France?<turn|>
<|turn>model



In [6]:
result = mlx_vlm.generate(model, processor, chat, max_tokens=128)
print(result.text)

PARIS


## Thinking Generation

In [7]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=True,
)

print(chat)

<bos><|turn>system
<|think|>
<turn|>
<|turn>user
What is the capital of France?<turn|>
<|turn>model



In [8]:
result = mlx_vlm.generate(model, processor, chat, max_tokens=1024)
print(result.text)

<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The user is asking a factual question: "What is the capital of France?"
2.  **Retrieve Knowledge:** Access internal knowledge base for geography/world capitals.
3.  **Determine the Answer:** The capital of France is Paris.
4.  **Formulate the Response:** State the answer clearly and concisely. (Self-Correction/Review: The answer is straightforward and correct.)<channel|>The capital of France is **Paris**.


## Multi-Turn Generation

In [9]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
)
response = mlx_vlm.generate(model, processor, chat, max_tokens=128).text

print(response)

The capital of France is **Paris**.


In [10]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'}]


In [11]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [12]:
chat = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
)
response = mlx_vlm.generate(model, processor, chat, max_tokens=128).text

print(response)

There are many famous landmarks in Paris, but perhaps the most iconic one is the **Eiffel Tower**.


## Streaming Generation

In [13]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
)

for result in mlx_vlm.stream_generate(model, processor, chat, max_tokens=128):
    print(result.text, end="", flush=True)

The capital of France is **Paris**.